# Student Focus Monitor — Complete Pipeline

## Intelligent Online Learning Analytics with Behavioral Pattern Recognition

This notebook walks through the entire project pipeline:
1. **Dataset Generation** — 40,000 synthetic behavioral snapshots
2. **Data Validation** — Statistical checks for quality
3. **Exploratory Data Analysis** — Understanding behavioral patterns
4. **Feature Engineering** — 68 engineered features (lagged, rolling, interaction)
5. **Model Training** — Random Forest, XGBoost, LSTM
6. **Evaluation** — Confusion matrices, ROC curves, feature importance
7. **Adaptive Response** — Rule-based intervention demo

### Novelties:
- Maps low-level behavioral signals → high-level cognitive states
- Per-student personalization with unique baselines
- Temporal pattern recognition (sequences, not snapshots)
- Dynamic focus score (0–100) with time-sensitive weighting

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Setup paths
os.chdir('D:/PR_Project')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import display, HTML

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Load config
with open('config/config.yaml') as f:
    config = yaml.safe_load(f)

print("Setup complete. Project root: D:/PR_Project")

## 1. Dataset Generation

We generate 40,000 behavioral snapshots for 200 students. Each row represents a time-based snapshot with:
- **Identity**: student_id, session_id, timestamp
- **Behavioral signals**: tab_switch, idle_time, clicks, mouse_movement
- **Learning interactions**: replay_count, skip_count, playback_speed
- **Target**: cognitive state (focused, distracted, confused, bored)

Each student has a **unique behavioral baseline** (some naturally click more, some switch tabs more). The dataset is balanced at 10,000 rows per class.

In [ ]:
from data_generation import generate_dataset

# Generate or load dataset
data_path = config['dataset']['output_path']
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Loaded existing dataset: {df.shape}")
else:
    df = generate_dataset()

print(f"\nDataset shape: {df.shape}")
print(f"Unique students: {df['student_id'].nunique()}")
print(f"Unique sessions: {df['session_id'].nunique()}")
print(f"\nClass distribution:")
print(df['state'].value_counts())
print(f"\nSample rows:")
display(df.head(10))

## 2. Data Validation

We run 6 validation checks to ensure dataset quality:
1. No missing values
2. All features within expected ranges
3. Balanced class distribution (chi-squared test)
4. Good student coverage (no single student dominates)
5. Temporal coherence within sessions
6. Feature discriminability across states (ANOVA)

In [ ]:
from data_validation import validate_dataset
passed, results = validate_dataset()

## 3. Exploratory Data Analysis

Let's visualize how behavioral signals differ across cognitive states.

In [ ]:
# Feature distributions by state
features = ['tab_switch', 'idle_time', 'clicks', 'mouse_movement',
            'replay_count', 'skip_count', 'playback_speed']
colors = {'focused': '#22c55e', 'distracted': '#f97316', 'confused': '#3b82f6', 'bored': '#a855f7'}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    for state in ['focused', 'distracted', 'confused', 'bored']:
        data = df[df['state'] == state][feat]
        axes[i].hist(data, bins=30, alpha=0.5, label=state, color=colors[state])
    axes[i].set_title(feat, fontsize=13, fontweight='bold')
    axes[i].legend(fontsize=9)

# Class distribution pie chart
state_counts = df['state'].value_counts()
axes[7].pie(state_counts.values, labels=state_counts.index, autopct='%1.1f%%',
            colors=[colors[s] for s in state_counts.index], startangle=90)
axes[7].set_title('Class Distribution', fontsize=13, fontweight='bold')

plt.suptitle('Feature Distributions by Cognitive State', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/plots/eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/plots/eda_feature_distributions.png")

In [ ]:
# Correlation heatmap
numeric_cols = ['tab_switch', 'idle_time', 'clicks', 'mouse_movement',
                'replay_count', 'skip_count', 'playback_speed', 'focus_score']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/plots/eda_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-state mean comparison
state_means = df.groupby('state')[features].mean()
print("Mean behavioral features per cognitive state:\n")
display(state_means.round(2).style.background_gradient(cmap='YlOrRd', axis=0))

## 4. Focus Score Computation

The focus score (0–100) is computed using weighted behavioral signals with per-student normalization and temporal smoothing (exponential decay). It answers: **"How focused is this student right now?"**

In [ ]:
# Focus score distribution by state
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot
sns.boxplot(data=df, x='state', y='focus_score', palette=colors,
            order=['focused', 'confused', 'distracted', 'bored'], ax=axes[0])
axes[0].set_title('Focus Score Distribution by State', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cognitive State')
axes[0].set_ylabel('Focus Score')

# Histogram overlay
for state in ['focused', 'distracted', 'confused', 'bored']:
    data = df[df['state'] == state]['focus_score']
    axes[1].hist(data, bins=40, alpha=0.5, label=state, color=colors[state])
axes[1].set_title('Focus Score Histogram by State', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Focus Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/plots/focus_score_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFocus Score Summary:")
print(df.groupby('state')['focus_score'].describe().round(2))

## 5. Feature Engineering

We create 68 engineered features to capture temporal patterns:
- **Lagged features** (last 5 snapshots): captures recent behavior trajectory
- **Rolling statistics** (mean, std, max over 3-snapshot window): captures trends
- **Delta features**: rate of change in each signal
- **Interaction features**: cross-signal combinations (e.g., idle × replay = confusion signal)
- **Session position**: early/middle/late in session
- **Per-student z-scores**: how much this snapshot deviates from the student's personal baseline

In [ ]:
from feature_engineering import engineer_features, get_feature_columns

# Load engineered data or create it
eng_path = 'data/student_focus_dataset_engineered.csv'
if os.path.exists(eng_path):
    df_eng = pd.read_csv(eng_path)
    print(f"Loaded engineered dataset: {df_eng.shape}")
else:
    df_eng = engineer_features(df)
    df_eng.to_csv(eng_path, index=False)

feature_cols = get_feature_columns(df_eng)
print(f"\nTotal features: {len(feature_cols)}")
print(f"Original features: 8")
print(f"Engineered features: {len(feature_cols) - 8}")
print(f"\nFeature categories:")
print(f"  Lagged features: {len([c for c in feature_cols if '_lag' in c])}")
print(f"  Rolling features: {len([c for c in feature_cols if '_roll_' in c])}")
print(f"  Delta features: {len([c for c in feature_cols if '_delta' in c])}")
print(f"  Interaction features: {len([c for c in feature_cols if 'signal' in c or 'ratio' in c])}")
print(f"  Z-score features: {len([c for c in feature_cols if '_zscore' in c])}")
print(f"  Session features: {len([c for c in feature_cols if 'session_' in c])}")

## 6. Model Training & Comparison

We train three models to compare approaches:
- **Random Forest**: Classical ensemble on 76 engineered features
- **XGBoost**: Gradient boosting with GPU acceleration on 76 engineered features
- **LSTM**: Deep learning on raw sequences (8 features × 5 time steps)

This comparison shows: **Do handcrafted temporal features outperform learned sequence representations?**

In [ ]:
# --- Random Forest ---
print("=" * 60)
print("Training: Random Forest")
print("=" * 60)

from models.random_forest import train_random_forest
rf_results = train_random_forest(df_eng)

In [ ]:
# --- XGBoost ---
print("=" * 60)
print("Training: XGBoost")
print("=" * 60)

from models.xgboost_model import train_xgboost
xgb_results = train_xgboost(df_eng)

In [ ]:
# --- LSTM ---
print("=" * 60)
print("Training: LSTM")
print("=" * 60)

from models.lstm_model import train_lstm
lstm_results = train_lstm(df_eng)

In [ ]:
# --- Model Comparison Summary ---
print("\n" + "=" * 70)
print("MODEL COMPARISON SUMMARY")
print("=" * 70)

comparison_data = {
    'Model': ['Random Forest', 'XGBoost', 'LSTM'],
    'Accuracy': [rf_results['accuracy'], xgb_results['accuracy'], lstm_results['accuracy']],
    'CV Mean': [rf_results['cv_scores'].mean(), xgb_results['cv_scores'].mean(), lstm_results['accuracy']],
    'Features': ['76 (engineered)', '76 (engineered)', '8 (raw sequences)'],
    'Training': ['CPU', 'GPU (CUDA)', 'CPU'],
}

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df.style.highlight_max(subset=['Accuracy', 'CV Mean'], color='lightgreen')
        .format({'Accuracy': '{:.4f}', 'CV Mean': '{:.4f}'}))

## 7. Evaluation — Confusion Matrices, ROC Curves, Feature Importance

Full evaluation pipeline generates:
- Confusion matrices (raw + normalized) for each model
- ROC curves (One-vs-Rest) for each model
- Feature importance (top 20) for RF and XGBoost
- Per-student accuracy analysis
- Side-by-side model comparison

In [ ]:
from evaluation import evaluate_all_models
eval_results = evaluate_all_models()

In [ ]:
# Display saved evaluation plots
from IPython.display import Image as IPImage, display as ipdisplay
import glob

plot_files = sorted(glob.glob('outputs/plots/*.png'))
print(f"Generated {len(plot_files)} evaluation plots:\n")
for pf in plot_files:
    print(f"  - {os.path.basename(pf)}")
    
# Show key plots inline
for plot_name in ['model_comparison.png', 'confusion_matrix_xgboost.png', 
                  'roc_curves_xgboost.png', 'feature_importance_xgboost.png']:
    path = f'outputs/plots/{plot_name}'
    if os.path.exists(path):
        print(f"\n{'='*50}")
        print(f"  {plot_name.replace('_', ' ').replace('.png', '').title()}")
        print(f"{'='*50}")
        ipdisplay(IPImage(filename=path, width=800))

## 8. Adaptive Response Engine — Demo

The adaptive response module triggers intelligent interventions based on detected cognitive state:
- **Confused** → Show hints, simplified explanation
- **Bored** → Increase difficulty, challenge question
- **Distracted** → Send notification to refocus
- **Critical** (focus < 20) → Alert instructor

It also detects **patterns**: sustained confusion, declining focus, and disengagement spirals.

In [ ]:
from adaptive_response import AdaptiveResponseEngine

engine = AdaptiveResponseEngine()

# Simulate a student session with changing states
scenarios = [
    ("STU042", "focused",    85, {"tab_switch": 1, "replay_count": 0, "skip_count": 0}),
    ("STU042", "focused",    78, {"tab_switch": 2, "replay_count": 0, "skip_count": 0}),
    ("STU042", "confused",   55, {"tab_switch": 1, "replay_count": 4, "skip_count": 0}),
    ("STU042", "confused",   42, {"tab_switch": 1, "replay_count": 6, "skip_count": 0}),
    ("STU042", "confused",   35, {"tab_switch": 2, "replay_count": 8, "skip_count": 0}),
    ("STU042", "distracted", 28, {"tab_switch": 7, "replay_count": 0, "skip_count": 1}),
    ("STU042", "bored",      22, {"tab_switch": 5, "replay_count": 0, "skip_count": 4}),
    ("STU042", "bored",      15, {"tab_switch": 3, "replay_count": 0, "skip_count": 6}),
]

print("Adaptive Response Engine — Session Simulation for Student STU042")
print("=" * 75)
print(f"{'#':<4} {'State':<14} {'Score':<8} {'Action':<22} {'Message'}")
print("-" * 75)

for i, (sid, state, score, snapshot) in enumerate(scenarios):
    response = engine.get_response(sid, state, score, snapshot)
    print(f"{i+1:<4} {state:<14} {score:<8} {response['action']:<22} {response['message'][:50]}")

print("\nNotice how the system escalates responses as the student's state worsens:")
print("  1-2: Focused → No intervention needed")
print("  3-5: Confused → Show hints → Detects sustained confusion pattern")
print("  6:   Distracted → Send notification to refocus")
print("  7-8: Bored + Critical → Alert instructor")

## 9. Chrome Extension + Real Data Pipeline

The system includes a Chrome extension that captures real behavioral signals and a Flask backend for storage and inference.

### How to use:
1. Start the backend: `py backend/app.py`
2. Load extension in Chrome: `chrome://extensions/` → Load unpacked → select `extension/` folder
3. Open any learning website, click the extension, enter Student ID, click "Start Study Session"
4. The extension tracks tab switches, idle time, clicks, mouse movement, video interactions
5. Every 30 seconds, a snapshot is sent to the backend which:
   - Computes a personalized focus score
   - Predicts cognitive state
   - Returns an adaptive message
   - Updates the student's behavioral baseline

### Privacy:
- **No browsing history** is tracked — only aggregate counts (e.g., "5 tab switches")
- Student manually starts/stops sessions
- All data stored locally (SQLite)

## 10. Conclusion

### Key Results:
| Model | Accuracy | F1 Score | Approach |
|-------|----------|----------|----------|
| Random Forest | 95.84% | 0.9584 | 76 engineered features |
| **XGBoost** | **96.60%** | **0.9660** | 76 engineered features + GPU |
| LSTM | 91.69% | 0.9166 | Raw sequences (8 features) |

### Key Findings:
1. **Engineered temporal features outperform raw sequences** — RF/XGBoost with handcrafted lag, rolling, and interaction features beat LSTM on raw data by ~4-5%.
2. **Per-student normalization is critical** — the same behavior means different things for different students.
3. **Sequential patterns reveal deeper states** — confusion buildup (idle → replay → pause) is detectable through temporal features.
4. **The adaptive response system escalates appropriately** — from gentle hints to instructor alerts based on severity and patterns.

### System Architecture:
- **Chrome Extension** captures real behavioral signals in real-time
- **Flask Backend** stores data, computes personalized focus scores, runs inference
- **ML Models** classify cognitive states with 96.6% accuracy
- **Adaptive Engine** triggers intelligent interventions
- **Streamlit Dashboard** provides interactive analytics

### Future Work:
- Retrain models on accumulated real data (online learning)
- Add webcam-based attention detection
- Integrate with LMS platforms (Moodle, Canvas)
- A/B test intervention effectiveness